# 📖 Notebook 2: Activity Feed & Social Features

Strava isn't just about tracking your own runs — it's about seeing what your friends are doing.
In this notebook we build the **social feed**: querying friends' activities, pagination,
and caching feeds in Redis to avoid hammering the database.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **bi-directional friendships** work in a database
- How to query a **friends activity feed** with pagination
- Why feed queries get expensive at scale
- How to cache feeds in **Redis** for fast reads
- The difference between **fan-out-on-write** and **fan-out-on-read**

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/strava
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `strava_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "strava_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 👥 Friendships: A Bi-Directional Graph

In Strava, friendships are **mutual** — if Alice follows Bob, Bob also follows Alice.

In the database, we store this with **two rows** per friendship:

```
friends table:
  user_id=1, friend_id=2   ← Alice → Bob
  user_id=2, friend_id=1   ← Bob → Alice
```

This makes queries simple: "give me all friends of user X" is just
`SELECT friend_id FROM friends WHERE user_id = X`.

The trade-off is we store 2× the rows, but friendship tables are small.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Who are Alice's friends?
cur.execute("""
    SELECT u.id, u.username, u.display_name, u.city
    FROM friends f
    JOIN users u ON f.friend_id = u.id
    WHERE f.user_id = 1
    ORDER BY u.username;
""")
friends = cur.fetchall()

print(f"Alice's friends ({len(friends)} total):")
print(f"{'ID':>4} {'Username':<10} {'Name':<20} {'City':<15}")
print("-" * 55)
for f in friends:
    print(f"{f['id']:>4} {f['username']:<10} {f['display_name']:<20} {f['city']:<15}")

conn.close()

## 📰 The Activity Feed: Two Modes

Strava's feed has two views:

1. **My Activities** (`mode=USER`) — your own completed runs and rides
2. **Friends Feed** (`mode=FRIENDS`) — completed activities from all your friends

Let's implement both with pagination.

In [ ]:
def get_activity_feed(user_id, mode='USER', page=1, page_size=5):
    """
    Fetch a page of activities.
    
    mode='USER'    → the user's own activities
    mode='FRIENDS' → activities from the user's friends
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    offset = (page - 1) * page_size
    
    if mode == 'USER':
        cur.execute("""
            SELECT a.id, u.username, a.type, a.title,
                   ROUND(a.distance_m::numeric) as distance_m,
                   a.duration_s, a.completed_at
            FROM activities a
            JOIN users u ON a.user_id = u.id
            WHERE a.state = 'COMPLETE'
              AND a.user_id = %s
            ORDER BY a.completed_at DESC
            LIMIT %s OFFSET %s;
        """, (user_id, page_size, offset))
    else:
        # Friends feed: activities from all friends
        cur.execute("""
            SELECT a.id, u.username, a.type, a.title,
                   ROUND(a.distance_m::numeric) as distance_m,
                   a.duration_s, a.completed_at
            FROM activities a
            JOIN users u ON a.user_id = u.id
            WHERE a.state = 'COMPLETE'
              AND a.user_id IN (
                  SELECT friend_id FROM friends WHERE user_id = %s
              )
            ORDER BY a.completed_at DESC
            LIMIT %s OFFSET %s;
        """, (user_id, page_size, offset))
    
    results = cur.fetchall()
    conn.close()
    return results

# Alice's own activities
print("=" * 70)
print("Alice's Activities (mode=USER)")
print("=" * 70)
my_feed = get_activity_feed(user_id=1, mode='USER', page=1)
for a in my_feed:
    print(f"  #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
          f"{a['distance_m']:>7} m  {a['duration_s']:>5} s  {a['title']}")

print()
print("=" * 70)
print("Alice's Friends Feed (mode=FRIENDS)")
print("=" * 70)
friends_feed = get_activity_feed(user_id=1, mode='FRIENDS', page=1)
for a in friends_feed:
    print(f"  #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
          f"{a['distance_m']:>7} m  {a['duration_s']:>5} s  {a['title']}")

In [ ]:
# Pagination demo — page through friends' activities
print("Paginating Alice's friends feed:")
print()

for page in range(1, 4):
    results = get_activity_feed(user_id=1, mode='FRIENDS', page=page, page_size=3)
    if not results:
        print(f"  Page {page}: (empty — no more activities)")
        break
    print(f"  Page {page}:")
    for a in results:
        print(f"    #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
              f"{a['distance_m']:>7} m  {a['title']}")
    print()

print("💡 Pagination uses LIMIT/OFFSET. At scale, cursor-based pagination")
print("   (WHERE completed_at < last_seen_timestamp) performs better.")

## ⚡ The Problem: Feed Queries Are Expensive

Look at our friends feed query — it does:
1. A subquery to find all friend IDs
2. An `IN (...)` filter across all activities
3. A sort by completion time

With millions of users and activities, this gets slow fast.

Let's measure the cost.

In [ ]:
# Measure query time for the friends feed
conn = get_db()
cur = conn.cursor()

times_db = []
for _ in range(50):
    start = time.time()
    cur.execute("""
        SELECT a.id, u.username, a.type, a.title,
               a.distance_m, a.duration_s, a.completed_at
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
          AND a.user_id IN (
              SELECT friend_id FROM friends WHERE user_id = 1
          )
        ORDER BY a.completed_at DESC
        LIMIT 10;
    """)
    cur.fetchall()
    times_db.append((time.time() - start) * 1000)

conn.close()

avg_db = sum(times_db) / len(times_db)
print(f"Friends feed from PostgreSQL (50 queries):")
print(f"  Average: {avg_db:.2f} ms")
print(f"  Min:     {min(times_db):.2f} ms")
print(f"  Max:     {max(times_db):.2f} ms")
print()
print("💡 Seems fast with our small dataset.")
print("   But imagine 100M users with 500 friends each — this blows up.")
print("   That's where caching comes in.")

## 🚀 Caching Feeds in Redis

The idea: cache each user's friends feed in Redis so we skip the database entirely.

**Pattern: Cache-Aside**
1. Check Redis for the feed → if found, return it (**cache hit**)
2. If not in Redis → query Postgres → store result in Redis with a TTL
3. When a friend completes an activity → **invalidate** the cached feed

```
Redis key: feed:friends:{user_id}
Value:     JSON array of recent activities
TTL:       60 seconds (feeds are slightly stale, that's OK)
```

In [ ]:
r = get_redis()

FEED_TTL = 60  # seconds

def get_friends_feed_cached(user_id, page=1, page_size=10):
    """
    Get friends feed with Redis caching.
    Returns (results, cache_hit: bool).
    """
    cache_key = f"feed:friends:{user_id}:page:{page}"

    # Step 1: Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # cache hit!

    # Step 2: Cache miss — query the database
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    offset = (page - 1) * page_size
    cur.execute("""
        SELECT a.id, u.username, a.type, a.title,
               ROUND(a.distance_m::numeric) as distance_m,
               a.duration_s
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
          AND a.user_id IN (
              SELECT friend_id FROM friends WHERE user_id = %s
          )
        ORDER BY a.completed_at DESC
        LIMIT %s OFFSET %s;
    """, (user_id, page_size, offset))
    results = cur.fetchall()
    conn.close()

    # Convert Decimal/datetime to JSON-friendly types
    serializable = []
    for row in results:
        serializable.append({
            'id': row['id'],
            'username': row['username'],
            'type': row['type'],
            'title': row['title'],
            'distance_m': float(row['distance_m']) if row['distance_m'] else 0,
            'duration_s': row['duration_s'],
        })

    # Step 3: Store in Redis with TTL
    r.setex(cache_key, FEED_TTL, json.dumps(serializable))

    return serializable, False  # cache miss

# First call: cache miss
feed, hit = get_friends_feed_cached(user_id=1)
print(f"Call 1: cache {'HIT ✅' if hit else 'MISS ❌'}  — {len(feed)} activities")

# Second call: cache hit!
feed, hit = get_friends_feed_cached(user_id=1)
print(f"Call 2: cache {'HIT ✅' if hit else 'MISS ❌'}  — {len(feed)} activities")

# Show the TTL
ttl = r.ttl("feed:friends:1:page:1")
print(f"\nTTL remaining: {ttl} seconds")
print("\n💡 Open RedisInsight at http://localhost:5540 to see the cached key!")

In [ ]:
# Measure the speed difference: cached vs uncached
r.delete("feed:friends:1:page:1")  # clear cache first

# Uncached (first call hits Postgres)
start = time.time()
for _ in range(50):
    r.delete("feed:friends:1:page:1")
    get_friends_feed_cached(user_id=1)
uncached_avg = ((time.time() - start) / 50) * 1000

# Cached (subsequent calls hit Redis)
get_friends_feed_cached(user_id=1)  # warm the cache
start = time.time()
for _ in range(50):
    get_friends_feed_cached(user_id=1)
cached_avg = ((time.time() - start) / 50) * 1000

print(f"Feed latency comparison (50 calls each):")
print(f"  Uncached (Postgres): {uncached_avg:.2f} ms")
print(f"  Cached (Redis):      {cached_avg:.2f} ms")
print(f"  Speedup:             {uncached_avg/cached_avg:.1f}×")
print()
print("💡 At scale with millions of activities, the speedup is even more dramatic.")

## 🔄 Cache Invalidation: When a Friend Completes an Activity

When Bob finishes a run, Alice's cached feed becomes **stale** — it doesn't include
Bob's new activity. We need to invalidate it.

**Strategy**: when any user completes an activity, delete all their friends' cached feeds.

```
Bob completes a run
  → Find all of Bob's friends: [Alice, Carol, Frank, ...]
  → Delete keys: feed:friends:1:*, feed:friends:3:*, feed:friends:6:*, ...
  → Next time Alice opens her feed, it rebuilds from Postgres
```

In [ ]:
def on_activity_completed(user_id):
    """
    Called when a user completes an activity.
    Invalidates all friends' cached feeds so they see the new activity.
    """
    conn = get_db()
    cur = conn.cursor()

    # Find all friends of this user
    cur.execute("SELECT friend_id FROM friends WHERE user_id = %s", (user_id,))
    friend_ids = [row[0] for row in cur.fetchall()]
    conn.close()

    # Delete each friend's cached feed
    invalidated = 0
    for fid in friend_ids:
        # Delete all pages of this friend's feed cache
        keys = r.keys(f"feed:friends:{fid}:page:*")
        if keys:
            r.delete(*keys)
            invalidated += len(keys)

    return friend_ids, invalidated

# Demo: Bob (user 2) completes a new activity
# First, make sure Alice's feed is cached
get_friends_feed_cached(user_id=1)
print(f"Alice's feed cached: {r.exists('feed:friends:1:page:1')} (1=yes)")

# Bob completes a run
friend_ids, count = on_activity_completed(user_id=2)
print(f"\nBob completed a run!")
print(f"  Friends notified: {friend_ids}")
print(f"  Cache keys invalidated: {count}")

# Alice's cache should be gone now
print(f"\nAlice's feed cached: {r.exists('feed:friends:1:page:1')} (0=no)")
print("\n💡 Next time Alice opens her feed, it will rebuild from Postgres")
print("   and include Bob's new activity.")

## 🤔 Fan-Out-on-Read vs Fan-Out-on-Write

There are two main ways to build a social feed:

### Fan-Out-on-Read (what we built above)
When Alice opens her feed, we query all her friends' activities on the spot.

```
Alice opens feed
  → Find Alice's friends
  → Query: SELECT ... WHERE user_id IN (friends) ORDER BY time
  → Return results
```

**Pros**: Simple, always fresh  
**Cons**: Slow at scale (many JOINs per request)

### Fan-Out-on-Write (Twitter/Instagram style)
When Bob completes a run, we immediately write it to all his friends' feeds.

```
Bob completes a run
  → Find Bob's friends: [Alice, Carol, Frank]
  → Append to each friend's feed list in Redis:
      LPUSH feed:alice {...}
      LPUSH feed:carol {...}
      LPUSH feed:frank {...}
```

**Pros**: Reads are instant (just read from pre-built list)  
**Cons**: Writes are expensive (celebrity with 10M followers = 10M writes)

### Which to Use?

| Scenario | Best Approach |
|----------|---------------|
| Most users have < 500 friends | Fan-out-on-write |
| Some users have millions of followers | Hybrid (fan-out-on-write for normal users, fan-out-on-read for celebrities) |
| Strava (fitness app, ~100-500 friends) | Either works — fan-out-on-read + cache is simpler |

For Strava, fan-out-on-read with caching (what we built) is a great fit because:
- Friend counts are small (hundreds, not millions)
- Activities happen at most a few times per day
- Slight staleness is acceptable

In [ ]:
# Let's also demonstrate fan-out-on-write to compare

def fanout_on_write(user_id, activity_data):
    """
    When a user completes an activity, push it to all friends' feed lists in Redis.
    """
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT friend_id FROM friends WHERE user_id = %s", (user_id,))
    friend_ids = [row[0] for row in cur.fetchall()]
    conn.close()

    payload = json.dumps(activity_data)
    for fid in friend_ids:
        r.lpush(f"feed:fanout:{fid}", payload)
        r.ltrim(f"feed:fanout:{fid}", 0, 49)  # keep only last 50

    return len(friend_ids)

def read_fanout_feed(user_id, count=5):
    """Read a pre-built feed from Redis (fan-out-on-write)."""
    items = r.lrange(f"feed:fanout:{user_id}", 0, count - 1)
    return [json.loads(item) for item in items]

# Bob completes a run → fan out to all friends
activity = {
    'id': 999, 'username': 'bob', 'type': 'RUN',
    'title': 'Evening Run', 'distance_m': 5200, 'duration_s': 1500
}
written_to = fanout_on_write(user_id=2, activity_data=activity)
print(f"Bob completed 'Evening Run' → fanned out to {written_to} friends")
print()

# Alice reads her pre-built feed
alice_feed = read_fanout_feed(user_id=1)
print("Alice's fan-out feed (from Redis list):")
for item in alice_feed:
    print(f"  {item['username']}: {item['title']} — {item['distance_m']} m")
print()
print("💡 No database query needed! The feed was pre-built when Bob completed his run.")

## 📊 Real-Time Friend Tracking (Polling)

A common follow-up question: *"Can friends watch each other's runs in real-time?"*

The key insight from the source material: **polling is better than WebSockets here**.

Why?
- Updates arrive every 2–5 seconds (predictable interval)
- A few seconds of delay is perfectly acceptable
- Polling is simpler to implement and scale

```
Bob is running:
  Every 5s → phone sends GPS update to server → stored in Redis

Alice is watching:
  Every 7s → phone polls server → "Where is Bob now?" → Redis lookup
```

We offset Alice's polling by a few seconds to ensure data is available.

In [ ]:
import random

def simulate_live_tracking():
    """Simulate Bob running and Alice watching in real-time."""
    # Bob starts running
    lat, lon = 37.7955, -122.3935

    print("🏃 Simulating live tracking:")
    print("   Bob is running. Alice is watching.")
    print()

    for t in range(0, 25, 5):
        # Bob's phone sends GPS update to Redis
        lat -= random.uniform(0.0003, 0.0006)
        lon += random.uniform(0.0003, 0.0006)
        location = {'lat': round(lat, 4), 'lon': round(lon, 4), 'time': t}
        r.set('live:activity:bob', json.dumps(location), ex=30)

        # Alice's phone polls after a short delay
        time.sleep(0.1)  # simulated network delay
        data = json.loads(r.get('live:activity:bob'))

        print(f"  t={t:>2}s  Bob sends: ({data['lat']}, {data['lon']})  "
              f"→ Alice sees: ({data['lat']}, {data['lon']})")

    r.delete('live:activity:bob')
    print()
    print("💡 Simple polling works great when updates are predictable.")
    print("   No need for WebSocket complexity!")

simulate_live_tracking()

## 🧹 Cleanup

In [ ]:
# Clean up all Redis keys we created
r = get_redis()
for pattern in ['feed:friends:*', 'feed:fanout:*', 'live:*']:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)
        print(f"🧹 Deleted {len(keys)} keys matching '{pattern}'")
print("🧹 Done!")

## 📚 Summary

### Key Takeaways

1. **Bi-directional friendships** use two rows per pair for simple queries
2. **Friends feed** uses a subquery + JOIN — simple but gets expensive at scale
3. **Cache-aside with Redis** gives huge speedups for feed reads
4. **Cache invalidation** on activity completion keeps feeds fresh
5. **Fan-out-on-write** pre-builds feeds but costs more on writes
6. **Polling** (not WebSockets) is the right choice for live tracking with predictable updates

### Next Up

In **Notebook 3**, we tackle **route matching and segment leaderboards** —
detecting when a GPS trace crosses a famous segment and ranking athletes
with Redis Sorted Sets.